In [5]:
#!pip install qiskit-machine-learning
#!pip install qiskit_algorithms

import numpy as np
import matplotlib.pyplot as plt
import scipy.io
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.svm import SVR
from sklearn.gaussian_process.kernels import RBF
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from datetime import datetime
import pandas as pd

#Qiskit 1.0

#qiskit_algorithms
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_algorithms.optimizers import SPSA, Optimizer, OptimizerResult, Minimizer

#qiskit_circuits
from qiskit.circuit import Parameter, ParameterVector, ParameterExpression
from qiskit.circuit.library import ZZFeatureMap, PauliFeatureMap
from qiskit.circuit.parameterexpression import ParameterValueType

#qiskit_machine_learning
from qiskit_machine_learning.algorithms import QSVR
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.kernels import TrainableFidelityQuantumKernel
from qiskit_machine_learning.kernels.algorithms import QuantumKernelTrainer
from qiskit_machine_learning.algorithms import QSVC

#qiskit_others

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.providers import Backend
from qiskit.result import Result
from qiskit.primitives import Sampler, BaseSampler
from qiskit.visualization import circuit_drawer


# Geração dos pontos (x,y)

In [6]:
N = 10
x = [i/100 for i in range(N)] 
def makeData(x):    
    r = [a/3 for a in x]
    y = np.sin(x)+np.cos(x) + np.random.uniform(-.5, 1.5, len(x))
    return np.array(y+r)

y = makeData(x)

#Aumentando o número de caracterísiticas de x
#A parte quântica só roda com dim(x)>=2. Por isso estou completando com zero.
x=[[x[i],0] for i in range(N)]
x = np.array(x).reshape(-2,2)
# plt.scatter(x, y, s=2, color="blue")
# plt.show()
print(x)


[[0.   0.  ]
 [0.01 0.  ]
 [0.02 0.  ]
 [0.03 0.  ]
 [0.04 0.  ]
 [0.05 0.  ]
 [0.06 0.  ]
 [0.07 0.  ]
 [0.08 0.  ]
 [0.09 0.  ]]


# SVRs Clássicos

In [7]:
svr = SVR(kernel='linear').fit(x, y) # kernel=rbf é o padrão
yfit = svr.predict(x)
mse_linear=mean_squared_error(y, yfit)

svr = SVR(kernel='poly').fit(x, y) # kernel=rbf é o padrão
yfit = svr.predict(x)
mse_poly=mean_squared_error(y, yfit)


svr = SVR(kernel='rbf').fit(x, y) # kernel=rbf é o padrão
yfit = svr.predict(x)
mse_rbf=mean_squared_error(y, yfit)

# Definindo algumas funções de mapeamento para kernel quântico

In [8]:
###################### quantum feature map ###########################
#As funções abaixo definem funções de kernel distintas.
def phi_1(x):
    if len(x) == 1:
        coeff = x[0]
    else:
        coeff = (np.pi - x[0])*(np.pi-x[1])
    return coeff

def phi_2(x):
    if len(x) == 1:
        coeff = x[0]
    else:
        coeff = np.pi*x[0]*x[1]
    return coeff

def phi_3(x):
    if len(x) == 1:
        coeff = x[0]
    else:
        coeff = np.pi/(np.cos(x[0])*np.cos(x[1]))
    return coeff

def phi_4(x):
    if len(x) == 1:
        coeff = x[0]
    else:
        coeff = np.pi*np.cos(x[0])*np.cos(x[1])
    return coeff

def phi_5(x):
    if len(x) == 1:
        coeff = x[0]
    else:
        coeff = np.exp((abs(x[0]-x[1]))/(8/np.log(np.pi)))
    return coeff

def phi_6(x):
    if len(x) == 1:
        coeff = x[0]
    else:
        coeff = (np.pi/2)*(1 - x[0])*(1-x[1])
    return coeff

dict_of_encodings ={
       'map_func_1':phi_1,
       'map_func_2':phi_2,
       'map_func_3':phi_3,
       'map_func_4':phi_4,
       'map_func_5':phi_5,
       'map_func_6':phi_6,    
}

# Gerando uma lista de sequências de Pauli

In [9]:
#Cada sequência de matrizes de Pauli define um kernel distinto
list_paulis_=[['Z', 'ZZ'], ['X', 'Z', 'XX']]

# Quantum SVR

In [11]:
# sampler = Sampler()
# fidelity = ComputeUncompute(sampler=sampler)

# for function in dict_of_encodings:
#     for p in list_paulis_:
#         feature_map = PauliFeatureMap(feature_dimension=2, reps=2,entanglement="full", 
#                                       data_map_func=dict_of_encodings[function], paulis = p)
#         #feature_map = ZZFeatureMap(feature_dimension=2, reps=2, entanglement="linear")
#         qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)
#         qsvr = QSVR(quantum_kernel = qkernel)
#         qsvr.fit(x, y)
#         #score=qsvr.score(x, y)
#         yfit = qsvr.predict(x)
#         mse=mean_squared_error(y, yfit)
#         #print("R-squared:", score)
#         print(f"{function}, {p} -> MSE: {mse}")
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.state_fidelities import ComputeUncompute

sampler = StatevectorSampler()  # V2
fidelity = ComputeUncompute(sampler=sampler)

for function in dict_of_encodings:
    for p in list_paulis_:
        feature_map = PauliFeatureMap(
            feature_dimension=2,
            reps=2,
            entanglement="full",
            data_map_func=dict_of_encodings[function],
            paulis=p,
        )
        qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)
        qsvr = QSVR(quantum_kernel=qkernel)
        qsvr.fit(x, y)
        yfit = qsvr.predict(x)
        mse = mean_squared_error(y, yfit)
        print(f"{function}, {p} -> MSE: {mse}")


        
print(f'MSE Linear={mse_linear}')
print(f'MSE Polinomial={mse_poly}')
print(f'MSE RBF={mse_rbf}')

map_func_1, ['Z', 'ZZ'] -> MSE: 0.25669154576505215
map_func_1, ['X', 'Z', 'XX'] -> MSE: 0.2650776163963352
map_func_2, ['Z', 'ZZ'] -> MSE: 0.26011574836144924
map_func_2, ['X', 'Z', 'XX'] -> MSE: 0.258655627713362
map_func_3, ['Z', 'ZZ'] -> MSE: 0.25637960097102075
map_func_3, ['X', 'Z', 'XX'] -> MSE: 0.25669356967044
map_func_4, ['Z', 'ZZ'] -> MSE: 0.2587913915840523
map_func_4, ['X', 'Z', 'XX'] -> MSE: 0.25989119170439007
map_func_5, ['Z', 'ZZ'] -> MSE: 0.2578079058899534
map_func_5, ['X', 'Z', 'XX'] -> MSE: 0.2584439256348126
map_func_6, ['Z', 'ZZ'] -> MSE: 0.2574203200619149
map_func_6, ['X', 'Z', 'XX'] -> MSE: 0.2592367126767935
MSE Linear=0.2596555701209021
MSE Polinomial=0.2601014090590209
MSE RBF=0.2580361745595319
